In [18]:
# Setup
import importlib
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import pickle as pkl
import matplotlib.pyplot as plt

from sccoda.util import comp_ana as mod
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz

import sccoda.datasets as scd
from sccoda.model.scCODA_model import EricaModel

In [19]:
def run_hmc_once(
    data_matrix,
    covariate_matrix,
    cell_types,
    covariate_names,
    formula,
    ref_index,
    priors,
    hmc_params
):
    model = EricaModel(
        reference_cell_type=ref_index,
        data_matrix=data_matrix,
        covariate_matrix=covariate_matrix,
        cell_types=cell_types,
        covariate_names=covariate_names,
        formula=formula,
        **priors
    )

    res = model.sample_hmc(**hmc_params)
    return res


In [20]:
cov1000 = pd.read_csv("/users/ebrown62/scCODA/sccoda/datasets/scCODA_simulated_N=2000_K=5_P=19_SEED=0.csv")

df = cov1000.copy()

# Drop donor ID -- MUST DO
if "donor_id" in df.columns:
    df = df.drop(columns=["donor_id"])

cell_types = [c for c in df.columns if c.startswith("CT")]
covariates = [c for c in df.columns if c not in cell_types]

data_matrix = df[cell_types].values.astype(float)
covariate_matrix = df[covariates].values.astype(float)

ref_index = cell_types.index("CT5")  # or whatever reference
formula = "~ " + " + ".join(covariates)


In [21]:
priors = dict(
    alpha_loc=0.0,
    alpha_sd=5.0,
    sigma_hc_scale=1.0,
    gamma_loc=0.0,
    gamma_sd=1.0,
    tau_temperature=50.0
)
results = []

for step in [0.001, 0.002, 0.003, 0.005, 0.01]:
    for L in [5, 10, 20, 30]:
        
        hmc_params = dict(
            num_results=1000,
            num_burnin=200,
            step_size=step,
            num_leapfrog_steps=L
        )
        
        res = run_hmc_once(
            data_matrix,
            covariate_matrix,
            cell_types,
            covariate_names=covariates,
            formula=formula,
            ref_index=ref_index,
            priors=priors,
            hmc_params=hmc_params
        )

        acc = res.sample_stats["is_accepted"].values.mean()

        print(f"step={step}, L={L}, acc={acc:.3f}")
        
        results.append((step, L, acc))


Zero counts encountered in data! Added a pseudocount of 0.5.


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:10<00:00, 99.18it/s]


MCMC sampling finished. (13.890 sec)
Acceptance rate: 92.9%
step=0.001, L=5, acc=0.940


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:18<00:00, 52.70it/s]


KeyboardInterrupt: 

In [ ]:
import seaborn as sns

df = pd.DataFrame(results, columns=["step_size", "L", "acc"])
df_p = df.pivot("step_size", "L", "acc")
sns.heatmap(df_p, annot=True, cmap="viridis")

In [ ]:
cov5000 = pd.read_csv("/users/ebrown62/scCODA/sccoda/datasets/scCODA_simulated_N=5000_K=5_P=19_SEED=0.csv")

df2 = cov5000.copy()

if "donor_id" in df2.columns:
    df2 = df2.drop(columns=["donor_id"])
cell_types = [c for c in df2.columns if c.startswith("CT")]
covariates = [c for c in df2.columns if c not in cell_types]

data_matrix = df2[cell_types].values.astype(float)
covariate_matrix = df2[covariates].values.astype(float)

ref_index = cell_types.index("CT5")  
formula = "~ " + " + ".join(covariates)

In [ ]:
priors = dict(
    alpha_loc=0.0,
    alpha_sd=5.0,
    sigma_hc_scale=1.0,
    gamma_loc=0.0,
    gamma_sd=1.0,
    tau_temperature=50.0
)
results2 = []

for step in [0.001, 0.002, 0.003, 0.005, 0.01]:
    for L in [5, 10, 20, 30]:
        
        hmc_params = dict(
            num_results=1000,
            num_burnin=200,
            step_size=step,
            num_leapfrog_steps=L
        )
        
        res = run_hmc_once(
            data_matrix,
            covariate_matrix,
            cell_types,
            covariate_names=covariates,
            formula=formula,
            ref_index=ref_index,
            priors=priors,
            hmc_params=hmc_params
        )

        acc = res.sample_stats["is_accepted"].values.mean()

        print(f"step={step}, L={L}, acc={acc:.3f}")
        
        results2.append((step, L, acc))
